# 08 - Citations, Internationalization, External Context, and Summary

                This notebook keeps citation and diversity/internationalization analyses modular. It writes conservative outputs when citation or affiliation coverage is incomplete and generates the final analysis summary and reviewer-defense checklist.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

articles = pd.read_csv(ROOT / "outputs/tables/articles_clean.csv")
citations_path = ROOT / "citation_metadata_links.csv"
citations = pd.read_csv(citations_path) if citations_path.exists() else pd.DataFrame()
print(f"Citation rows: {len(citations):,}")

## Citation and Internationalization Metrics

In [ ]:
if citations.empty:
    citation_language = pd.DataFrame(columns=["LA", "n_records", "mean_citations", "median_citations"])
    affiliations = pd.DataFrame(columns=["citation_id", "affiliation", "country", "year"])
else:
    citations["Z9_numeric"] = pd.to_numeric(citations.get("Z9"), errors="coerce")
    citation_language = (
        citations.groupby("LA", dropna=False)
        .agg(
            n_records=("citation_id", "nunique"),
            mean_citations=("Z9_numeric", "mean"),
            median_citations=("Z9_numeric", "median"),
            n_matched_to_metadata=("meta_handle", lambda x: x.notna().sum()),
        )
        .reset_index()
    )

    country_aliases = {
        "Zealand": "New Zealand",
        "Tobago": "Trinidad and Tobago",
        "Republic": "Czech Republic",
        "Korea": "South Korea",
        "Emirates": "United Arab Emirates",
        "England": "United Kingdom",
        "Scotland": "United Kingdom",
        "Wales": "United Kingdom",
        "USA": "United States",
        "Africa": "South Africa",
    }
    affiliation_rows = []
    for row in citations.itertuples():
        c1 = compact_ws(getattr(row, "C1", ""))
        if not c1:
            continue
        parts = [part.strip() for part in c1.split("[") if part.strip()]
        for part in parts:
            country = part.split()[-1].strip(".,;)")
            country = country_aliases.get(country, country)
            affiliation_rows.append({"citation_id": row.citation_id, "affiliation": part, "country": country, "year": getattr(row, "year", pd.NA)})
    affiliations = pd.DataFrame(affiliation_rows)

write_csv(citation_language, ROOT / "outputs/tables/citation_language_metrics.csv")
write_csv(affiliations, ROOT / "outputs/tables/affiliations_extracted_from_citations.csv")

if not affiliations.empty:
    country_year = affiliations.groupby(["year", "country"]).size().reset_index(name="n_affiliations")
    totals = country_year.groupby("year")["n_affiliations"].transform("sum")
    country_year["share"] = country_year["n_affiliations"] / totals
    top_countries = affiliations["country"].value_counts().head(8).index
    plot_country = country_year[country_year["country"].isin(top_countries)]
else:
    country_year = pd.DataFrame(columns=["year", "country", "n_affiliations", "share"])
    plot_country = country_year

language_year = articles.groupby(["year", "language"]).size().reset_index(name="n_articles")
language_year["share"] = language_year["n_articles"] / language_year.groupby("year")["n_articles"].transform("sum")
internationalization = {
    "n_articles": len(articles),
    "n_citation_records": len(citations),
    "n_affiliation_rows_extracted": len(affiliations),
    "citation_metadata_coverage_share": float(citations["meta_handle"].notna().mean()) if not citations.empty and "meta_handle" in citations.columns else np.nan,
}
write_csv(pd.DataFrame([internationalization]), ROOT / "outputs/tables/internationalization_metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=language_year, x="year", y="share", hue="language", ax=axes[0])
axes[0].set_title("Publication language")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Share")
if not plot_country.empty:
    sns.lineplot(data=plot_country, x="year", y="share", hue="country", ax=axes[1])
axes[1].set_title("Affiliation countries from citation metadata")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Share of extracted affiliations")
axes[1].legend(loc="center left", bbox_to_anchor=(1, 0.5))
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_20_language_and_affiliation_over_time.png", dpi=220)
save_caption(ROOT, "fig_20_language_and_affiliation_over_time.png", "Publication language and extracted affiliation countries over time; affiliation coverage depends on available citation metadata.")
plt.show()
display(citation_language)

## Optional External Map Placeholders

In [ ]:
external_sample_path = ROOT / "outputs/tables/external_literature_sample.csv"
radiation_path = ROOT / "outputs/tables/radiation_power_clusters.csv"
if not external_sample_path.exists():
    write_csv(
        pd.DataFrame(
            columns=[
                "external_id",
                "title",
                "year",
                "source",
                "abstract",
                "embedding_status",
                "notes",
            ]
        ),
        external_sample_path,
    )
if not radiation_path.exists():
    write_csv(
        pd.DataFrame(
            columns=[
                "cluster_id",
                "cluster_label",
                "n_hsr_articles",
                "n_external_records",
                "interpretation_status",
                "notes",
            ]
        ),
        radiation_path,
    )
write_markdown(
    "# External Context Module\n\nNo external background map has been computed in this notebook. Use this module only when OpenAlex, Crossref, Web of Science, Scopus, or another comparison sample has sufficient coverage. Until then, phrase this as exploratory external semantic positioning, not radiation power.",
    ROOT / "outputs/diagnostics/external_context_status.md",
)

## Final Summary and Reviewer Defense Checklist

In [ ]:
def read_optional(path):
    path = ROOT / path
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

audit = read_optional("outputs/tables/corpus_audit.csv")
labels = read_optional("outputs/tables/topic_labels.csv")
innovation = read_optional("outputs/tables/topic_innovation_metrics.csv")
typology = read_optional("outputs/tables/person_typology.csv")
figures = sorted([p.name for p in (ROOT / "outputs/figures").glob("*.png")] + [p.name for p in (ROOT / "outputs/figures").glob("*.html")])

metric = dict(zip(audit.get("metric", []), audit.get("value", []))) if not audit.empty else {}
persistent_topics = innovation[innovation.get("is_stable_core", pd.Series(dtype=bool)).fillna(False)] if not innovation.empty else pd.DataFrame()

summary = [
    "# HSR Analysis Summary",
    "",
    f"- Corpus records with valid years: {metric.get('n_records_valid_year', 'not computed')}",
    f"- Articles with sufficient text: {metric.get('n_sufficient_text', 'not computed')}",
    f"- Core inclusion records: {metric.get('n_core_inclusion', 'not computed')}",
    f"- Candidate topics/clusters: {len(labels) if not labels.empty else 'not computed'}",
    f"- Persistent topics by operational heuristic: {len(persistent_topics) if not persistent_topics.empty else 'not computed'}",
    f"- Actor types assigned: {typology['actor_type'].nunique() if not typology.empty and 'actor_type' in typology else 'not computed'}",
    "",
    "## Generated Figures",
    "",
]
summary.extend(f"- `{figure}`" for figure in figures)
summary.extend(
    [
        "",
        "## Open Manual Steps",
        "",
        "- Fill `outputs/labels/topic_label_overrides.csv` with human topic labels and rationales.",
        "- Review `outputs/tables/top_persons_manual_check.csv` and add corrections to `data/name_corrections.csv`.",
        "- Fill `data/editors_roles.csv` with historically sourced editor and issue-editor roles.",
        "- Validate `outputs/labels/article_type_validation_sample.csv` before interpreting article-type trends.",
        "- Decide whether the optional external-context module has enough coverage to enter the main article.",
    ]
)
write_markdown("\n".join(summary), ROOT / "outputs/HSR_analysis_summary.md")

checklist = [
    "# Reviewer Defense Checklist",
    "",
    "## Why Embeddings Instead of LDA/STM?",
    "The pipeline uses multilingual sentence embeddings because the corpus spans German, English, and a small number of French records, and because the project aims to map semantic neighborhoods rather than infer latent topics as ontological entities. Clusters are described as candidate topics or semantic regions.",
    "",
    "## How Stable Are the Clusters?",
    "Cluster solutions are compared in `outputs/tables/topic_stability_metrics.csv` and visualized in `fig_model_stability_atlas.png`. Only stable regions should be discussed as robust findings.",
    "",
    "## How Was Guided Confirmation Avoided?",
    "Seed topics are used only in `outputs/tables/guided_vs_unguided_topic_comparison.csv` as a comparison against unguided embedding clusters. They do not determine the main cluster assignment.",
    "",
    "## How Were Language and Length Effects Checked?",
    "Language and text-length diagnostics are written to `outputs/tables/language_cluster_diagnostics.csv`, `outputs/tables/text_length_cluster_diagnostics.csv`, `fig_language_in_embedding_space.png`, and `fig_text_length_in_embedding_space.png`.",
    "",
    "## How Were Reviews, Editorials, and Bibliographies Treated?",
    "They are flagged in `outputs/tables/articles_clean.csv` and can be included or excluded through core/extended sensitivity analyses. They are not silently removed.",
    "",
    "## How Were Historical Editor Roles Reconstructed?",
    "Editor roles come from the manual, source-bearing file `data/editors_roles.csv`. If the file is empty or uncertain, editor analyses are treated as approximations.",
    "",
    "## Why Is Editorial Positioning Not a Causal Claim?",
    "The notebooks calculate co-occurrence, semantic position, issue concentration, and pre/post patterns. These are consistent with curatorial signatures but do not establish editor intention or causal mechanisms.",
    "",
    "## How Were LLM Labels Validated?",
    "The default notebook path uses transparent machine labels from evidence packs and requires manual overrides in `outputs/labels/topic_label_overrides.csv`. Any future LLM labeling should be saved with prompts and validation status.",
    "",
    "## Core vs Exploratory Analyses",
    "Core analyses: corpus audit, embeddings, clustering/stability, human-supervised labels, topic trends, person-topic profiles, issue concentration. Exploratory analyses: external field mapping, radiation-power language, and any diversity inference beyond aggregate language/affiliation patterns.",
    "",
    "## Claims to Avoid",
    "Avoid claims that models discover true topics, that UMAP point distances are metric intellectual distances, that editors caused topics, or that name-based demographic inference is definitive.",
]
write_markdown("\n".join(checklist), ROOT / "outputs/reviewer_defense_checklist.md")

print("Wrote outputs/HSR_analysis_summary.md and outputs/reviewer_defense_checklist.md")